# EM proofreading — Phase A (annotate) demo

Skeleton-driven fly-through over MICrONS minnie65: review a whole cell and drop
tagged annotations marking proofreading errors. **Read-only** — annotate now, edit
manually later (Phase B), then re-enter on the new root id (Phase C).

Design: [docs/proofreading-workflow.md](docs/proofreading-workflow.md) · vocabulary:
[CONTEXT.md](CONTEXT.md). Run in the `em` env (`uv run --extra em jupyter lab`, or pick
`.venv/bin/python3` as the VS Code kernel); needs a CAVE token at
`~/.cloudvolume/secrets/cave-secret.json`.


## 0. Imports


In [ ]:
import proofreading.em as em
from proofreading.em.wal import WAL

## 1. Connect to CAVE

`minnie65_public` is the read-only **sandbox** (no edits, no root changes). Use
`minnie65_phase3_v1` for the live, proofreadable datastack.


In [ ]:
client = em.EMClient('minnie65_public')
#   live: em.EMClient('minnie65_phase3_v1', version=<materialization_version>)
print('datastack', client.datastack, '| materialization', client.mat_version)

## 2. Start a session

Loads the L2 skeleton, captures the **seed supervoxel** (durable identity), builds the
viewer, and opens/append-resumes the write-ahead log. Playback knobs are explained in
section 5; the defaults below are the recommended ones.


In [ ]:
root_id = 864691135572530981   # example cell on minnie65_public
sess = em.ProofreadSession(
    client, root_id, wal_dir='./proofread_sessions',
    # --- playback (section 5) ---
    orient_to_path=False,           # native axis-aligned XYZ sections (fast); True = cross-section
    animate=True,                   # smooth glide between nodes
    seconds_per_step=0.5,           # glide duration per node
    dwell_seconds=1.5,              # rest at each node so the seg mask paints (see section 5)
    prefetch_window=3,              # warm upcoming nodes
    cross_section_render_scale=1.0, # 1.0 = full res (mip0); 2.0 ~ mip1 (faster)
)
print('seed supervoxel:', sess.seed)
print('branch paths:', len(sess.tree.branch_paths), '| summary:', sess.summary())

## 3. Open the viewer

Open this URL in a browser — the 4-panel layout (3 cross-sections + 3D). The target
cell is highlighted; other segments are off until you reveal them with `n`.

By default the fly-through keeps the **native axis-aligned XYZ sections** — they stream
fast (native chunk layout, like manual scrolling). To instead orient panel 1 as a
**cross-section ⊥ the neurite** (nicer for judging merges, but oblique slices stream
slower), set `orient_to_path=True` (§2) or `sess.set_orient_to_path(True)` live.


In [ ]:
sess.viewer

## 4. Control panel + key map

The panel shows the branch-path checklist with **Review**, **Mark done**, **Resolve
supervoxels**. While flying a path, use these keys in the neuroglancer window:

| key | action |
|-----|--------|
| `m` | merge error |
| `s` | split error |
| `e` | extend |
| `q` | question |
| `n` | toggle the segment under the cursor (reveal/hide a neighbor) |
| `x` | mark current branch path reviewed (and advance) |

A `merge error` ends the path early and **prunes the distal subtree** off the checklist.
Play / pause / step / reverse + the speed slider live on the FlyThrough controls shown after **Review**.


In [ ]:
sess.panel()

## 5. Playback (smooth glide + a rest where the seg paints)

Autoplay does a **smooth glide** to each node, then a brief **rest** there. The rest
isn't cosmetic: **neuroglancer only renders the segmentation when the camera is idle**,
so the mask paints during the rest. (Manual scrolling shows the seg because it has
natural idle gaps; continuous motion has none — so with `dwell_seconds=0` the seg only
appears when you pause.) ~2 s/node total lets the cache keep up.

- **`dwell_seconds`** (default 1.5): rest at each node so the seg paints. Lower it on a fast
  connection; `0` = pure continuous glide (seg only on manual pause).
- **`orient_to_path`** (default False): native axis-aligned sections (fast). True = cross-section ⊥ neurite (oblique, slower).
- **`animate`** (default True): smooth glide; False = jump-cut (instant move).
- **`seconds_per_step`**: glide duration per node (also the panel speed slider).
- **`prefetch_window`**: warms upcoming nodes so each rest starts with tiles already loading.
- **`cross_section_render_scale`**: `1.0` full-res mip0; `2.0` ≈ mip1 (~4× less EM data).
- **cache** (`gpu_memory_limit` / `system_memory_limit`): loaded tiles persist → reverse / revisit instant.
- **`load_gated`** (opt-in, default off): adaptively wait until GPU-resident instead of a fixed rest.

All adjustable **live**, no restart:


In [ ]:
sess.set_dwell(1.5)                      # rest at each node (lower if fast; 0 = continuous, seg on pause)
sess.set_orient_to_path(False)           # axis-aligned (fast) vs True for cross-section
sess.fly.set_speed(seconds_per_step=0.5) # glide speed (or use the panel slider)
sess.set_render_scale(2.0)               # ~mip1 while flying; 1.0 to inspect fine detail
sess.set_animate(True)                   # False = jump-cut

## 6. Or drive it from code

Instead of the panel buttons you can review paths programmatically.


In [ ]:
fly = sess.review_next()      # next to-review branch path; returns its FlyThrough
# fly.play() / fly.pause() / fly.step(1) / fly.set_speed(seconds_per_step=0.3)
fly

## 7. Checkpoint — resolve supervoxels

Annotations record the click `xyz` immediately (durable); supervoxels are derived in
batch from CloudVolume. Run at checkpoints / before stepping away.


In [ ]:
print('resolved', sess.resolve_supervoxels(), 'supervoxels')

## 8. Inspect what's recorded

The append-only write-ahead log is the source of truth — replay it any time.


In [ ]:
state = WAL.load(sess.wal.path)
print('log:', sess.wal.path)
for a in state.annotations.values():
    print(f'  {a.tag:12s} xyz={[round(c) for c in a.xyz]} supervoxel={a.supervoxel}')
print('coverage summary:', sess.summary())

## 9. After edits — re-entry (Phase C)

Once you've performed the manual splits/merges, the root id changes. Recover the current
root from the durable seed and start a fresh session: the same WAL resumes, prior coverage
re-attaches by L2 id, and edited regions fall back to `to_review`.

```python
new_root = client.current_root(sess.seed)
sess2 = em.ProofreadSession(client, new_root, wal_dir='./proofread_sessions')
sess2.summary()
```


## 10. Shutdown


In [ ]:
sess.close()